In [13]:
import numpy as np
from pyscf import gto, scf, lo, mp, cc
mol = gto.Mole()
mol.verbose = 4
mol.atom = '''
O   -1.485163346097   -0.114724564047    0.000000000000
H   -1.868415346097    0.762298435953    0.000000000000
H   -0.533833346097    0.040507435953    0.000000000000
O    1.416468653903    0.111264435953    0.000000000000
H    1.746241653903   -0.373945564047   -0.758561000000
H    1.746241653903   -0.373945564047    0.758561000000
'''
mol.basis = 'cc-pvdz'
mol.precision = 1e-10
mol.build()
mf = scf.RHF(mol).density_fit()
mf.kernel()

frozen = 2
mymp = mp.MP2(mf, frozen=frozen)
mymp.kernel()
efull_mp2 = mymp.e_corr
print(f'MP2 Corr = {efull_mp2:.8f}')

mycc = cc.CCSD(mf, frozen=frozen)
mycc.kernel()
efull_ccsd = mycc.e_corr
print(f'CCSD Corr = {efull_ccsd:.8f}')

efull_t = mycc.ccsd_t()
efull_ccsd_t = efull_ccsd + efull_t
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-23-generic', version='#23~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Apr 14 16:11:48 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Wed May  6 17:33:29 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 6
[INPUT] num. electrons = 20
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y               

In [2]:
print(f'MP2 Corr = {efull_mp2:.8f}')
print(f'CCSD Corr = {efull_ccsd:.8f}')
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

MP2 Corr = -0.40609899
CCSD Corr = -0.42461940
CCSD(T) Corr = -0.43105819


In [15]:
from pyscf.lno import LNOCCSD, LNOCCSD_T
from pyscf.lno.tools import autofrag_iao

# def test_lno_iao_by_thresh(self):
mol = mycc.mol
mf = mycc._scf
frozen = mycc.frozen

# IAO localization
orbocc = mf.mo_coeff[:,frozen:np.count_nonzero(mf.mo_occ)]
lo_iao = lo.iao.iao(mol, orbocc)
lo_iao = lo.orth.vec_lowdin(lo_iao, mf.get_ovlp())
moliao = lo.iao.reference_mol(mol)

frag_iao_list = autofrag_iao(moliao)

# gamma = 10
# threshs = [3e-5]
# # refs = [
# #     [-0.4054784012,-0.4240686326,-0.4303996712],
# #     [-0.4060479828,-0.4245745223,-0.4309965749],
# # ]

# emp2_iao = np.zeros(len(threshs))
# eccsd_iao = np.zeros(len(threshs))
# eccsd_t_iao = np.zeros(len(threshs))

# for i, thresh in enumerate(threshs):
#     mcc = LNOCCSD(mf, lo_coeff, frag_lolist, frozen=frozen).set(verbose=5)
#     mcc.lno_thresh = [thresh*gamma,thresh]
#     mcc.kernel()
#     emp2_iao[i] = mcc.e_corr_pt2
#     eccsd_iao[i] = mcc.e_corr_ccsd
#     eccsd_t_iao[i] = mcc.e_corr_ccsd_t

In [6]:
# from collections.abc import Iterable
from pyscf.lno import lnoccsd
from afqmc.lno_afqmc.mod_lnoccsd import lno_ccsd
from afqmc.lno_afqmc.prep import las_size, ao_comp, prep_afqmc_integral
from pyscf.lno.tools import autofrag_iao
from jax import random
import os

In [17]:
def run_lnoccsd(
        mf, 
        # options, # qmc parameters
        lo_coeff, 
        frag_lolist,
        nfrozen = 0, 
        thresh = 1e-6, 
        run_frg_list = None,
        atom_group = None, # = mol.elements if frag_lolist is grouped by atoms
        # chol_cut = 1e-5,
        ):

    mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=0)
    mlno.lno_thresh = [thresh*10,thresh]
    lno_thresh = mlno.lno_thresh
    nfrag = len(frag_lolist)
    lno_type = ['1h','1h']
    lno_pct_occ = [None, None]
    lno_norb = [[None,None]] * nfrag
    eris = mlno.ao2mo()

    # trial = options["trial"]

    if run_frg_list is None:
        run_frg_list = range(nfrag)

    run_frag_lolist = [frag_lolist[i] for i in run_frg_list]

    print(f'Number of LNO-FRAGMENT: {nfrag}')

    # seeds = random.randint(random.PRNGKey(options["seed"]), shape=(nfrag,), minval=0, maxval=100*nfrag)
    # options["max_error"] = options["max_error"]/np.sqrt(nfrag)
    emp2 = np.zeros(nfrag, dtype='float64')
    ecc  = np.zeros(nfrag, dtype='float64')
    nact = np.zeros(nfrag, dtype='int32')

    # Loop over fragment
    for ifrag, loidx in enumerate(run_frag_lolist):
        print("\n")
        width = 80
        msg = f" RUNNING LNO-FRAGMENT {run_frg_list[ifrag]+1}/{nfrag} "
        print(msg.center(width, '='))
        if atom_group is not None:
            print(f"Center Atom {atom_group[ifrag]}")

        orbloc = lo_coeff[:,loidx]
        lno_param = [{'thresh': lno_thresh[i], 'pct_occ': lno_pct_occ[i],
                        'norb': lno_norb[ifrag][i]} for i in [0,1]]

        lno_coeff, lno_frozen, uocc_loc, _ = mlno.make_las(eris, orbloc, lno_type, lno_param)

        nactocc, nactvir = las_size(mf, lno_frozen)
        print(f'number of active occupied orbitals: {nactocc}')
        print(f'number of active virtual orbitals: {nactvir}')

        # identify the center LO's AO component
        print(f'Locating Fragment {loidx} by AOs')
        ao_message = ao_comp(mf, orbloc)

        mo_occ = mlno.mo_occ
        lno_frozen, maskact = lnoccsd.get_maskact(lno_frozen, len(mo_occ))
        mcc = lnoccsd.CCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=3)
        mcc._s1e = mlno._s1e
        mcc._h1e = mlno._h1e
        mcc._vhf = mlno._vhf
        if mlno.kwargs_imp is not None:
            mcc = mcc.set(**mlno.kwargs_imp)
        # time0 = time.perf_counter()
        (eorb_mp2, eorb_cc), t1, t2 = \
            lno_ccsd(mcc, lno_coeff, uocc_loc, mo_occ, maskact) # <<< this is on CPU
        # time1 = time.perf_counter()

        print(f'LNO-MP2 Orbital Energy: {eorb_mp2:.8f}')
        print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')

        prjlo = uocc_loc @ uocc_loc.T.conj()
        # options["seed"] = seeds[ifrag]
        # _, _ = prep_afqmc(mf, lno_coeff, t1, t2, lno_frozen, prjlo, options, chol_cut=chol_cut)
        # run_lnoafqmc(options)
        # os.system(f'mv afqmc.out lnoafqmc.out{run_frg_list[ifrag]+1}')

        emp2[ifrag] = eorb_mp2
        ecc[ifrag]  = eorb_cc
        nact[ifrag] = nactocc + nactvir

    emp2_tot = sum(emp2)
    ecc_tot = sum(ecc)
    orbmax = max(nact)
    print(f'LNO-MP2 Corr Energy: {eorb_mp2:.8f}')
    print(f'LNO-CCSD Corr Energy: {eorb_cc:.8f}')
    print(f'Largest LNO Fragment: {orbmax}')
    return emp2_tot, ecc_tot, orbmax

In [9]:
frag_lolist

[[0, 1, 2, 3, 4], [5], [6], [7, 8, 9, 10, 11], [12], [13]]

In [10]:
run_frag_lolist

[[0, 1, 2, 3, 4], [5], [6], [7, 8, 9, 10, 11], [12], [13]]

In [34]:
lo_coeff = lo_iao
frag_lolist = frag_iao_list
nfrozen = 2
thresh = 3e-5
run_frg_list = [0]
atom_group = moliao.elements

In [ ]:
mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=0)
mlno.lno_thresh = [thresh*10,thresh]
lno_thresh = mlno.lno_thresh
nfrag = len(frag_lolist)
lno_type = ['1h','1h']
lno_pct_occ = [None, None]
lno_norb = [[None,None]] * nfrag
eris = mlno.ao2mo()

# trial = options["trial"]

if run_frg_list is None:
    run_frg_list = range(nfrag)

run_frag_lolist = [frag_lolist[i] for i in run_frg_list]

print(f'Number of LNO-FRAGMENT: {nfrag}')

# seeds = random.randint(random.PRNGKey(options["seed"]), shape=(nfrag,), minval=0, maxval=100*nfrag)
# options["max_error"] = options["max_error"]/np.sqrt(nfrag)
emp2 = np.zeros(nfrag, dtype='float64')
ecc  = np.zeros(nfrag, dtype='float64')
nact = np.zeros(nfrag, dtype='int32')

# Loop over fragment
for ifrag, loidx in enumerate(run_frag_lolist):
    print("\n")
    width = 80
    msg = f" RUNNING LNO-FRAGMENT {run_frg_list[ifrag]+1}/{nfrag} "
    print(msg.center(width, '='))
    if atom_group is not None:
        print(f"Center Atom {atom_group[ifrag]}")

    orbloc = lo_coeff[:,loidx]
    lno_param = None
    lno_param = [{'thresh': lno_thresh[i], 'pct_occ': lno_pct_occ[i],
                    'norb': lno_norb[ifrag][i]} for i in [0,1]]

    # uocc_loc = <lno_actocc|orbloc>
    lno_coeff, lno_frozen, uocc_loc, _ = mlno.make_las(eris, orbloc, lno_type, lno_param)

    nactocc, nactvir = las_size(mf, lno_frozen)
    print(f'number of active occupied orbitals: {nactocc}')
    print(f'number of active virtual orbitals: {nactvir}')

    # identify the center LO's AO component
    print(f'Locating Fragment {loidx} by AOs')
    ao_message = ao_comp(mf, orbloc)

    mo_occ = mlno.mo_occ
    lno_frozen, maskact = lnoccsd.get_maskact(lno_frozen, len(mo_occ))
    mcc = lnoccsd.CCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=3)
    mcc._s1e = mlno._s1e
    mcc._h1e = mlno._h1e
    mcc._vhf = mlno._vhf
    if mlno.kwargs_imp is not None:
        mcc = mcc.set(**mlno.kwargs_imp)
    # time0 = time.perf_counter()
    (eorb_mp2, eorb_cc), t1, t2 = \
        lno_ccsd(mcc, lno_coeff, uocc_loc, mo_occ, maskact) # <<< this is on CPU
    # time1 = time.perf_counter()

    print(f'LNO-MP2 Orbital Energy: {eorb_mp2:.8f}')
    print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')

    prjlo = uocc_loc @ uocc_loc.T.conj()
    # options["seed"] = seeds[ifrag]
    # _, _ = prep_afqmc(mf, lno_coeff, t1, t2, lno_frozen, prjlo, options, chol_cut=chol_cut)
    # run_lnoafqmc(options)
    # os.system(f'mv afqmc.out lnoafqmc.out{run_frg_list[ifrag]+1}')

    emp2[ifrag] = eorb_mp2
    ecc[ifrag]  = eorb_cc
    nact[ifrag] = nactocc + nactvir

emp2_tot = sum(emp2)
ecc_tot = sum(ecc)
orbmax = max(nact)
print(f'LNO-MP2 Corr Energy: {eorb_mp2:.8f}')
print(f'LNO-CCSD Corr Energy: {eorb_cc:.8f}')
print(f'Largest LNO Fragment: {orbmax}')

Number of LNO-FRAGMENT: 6


=========================== RUNNING LNO-FRAGMENT 1/6 ===========================
Center Atom O
number of active occupied orbitals: 5
number of active virtual orbitals: 24
Locating Fragment [0, 1, 2, 3, 4] by AOs
AOs with contribution > 0.01
        AO Label     Amp
      0 O 1s      0.9019
      0 O 2px     0.5103
      0 O 2py     0.4829
      0 O 2pz     0.4539
      0 O 2s      0.4206
      0 O 3pz     0.4033
      0 O 3py     0.3156
      0 O 3s      0.3042
      0 O 3px     0.2529
      2 H 2px     0.2257
      1 H 2py     0.2102
      1 H 2px     0.0825
      1 H 1s      0.0811
      2 H 1s      0.0769
      1 H 2pz     0.0722
      2 H 2pz     0.0700
      2 H 2py     0.0626
      2 H 2s      0.0360
      1 H 2s      0.0332
E(MODIFIED_CCSD) = -152.2904073592697  E_corr = -0.2279167123418015
LNO-MP2 Orbital Energy: -0.16955726
LNO-CCSD Orbital Energy: -0.17710627
LNO-MP2 Corr Energy: -0.16955726
LNO-CCSD Corr Energy: -0.17710627
Largest LNO Fragment: 2

In [68]:
dm_lno = mf.make_rdm1(lno_coeff, mf.mo_occ)
mf.kernel(dm0=dm_lno)



******** <class 'pyscf.df.df_jk.DFRHF'> ********
method = DFRHF
initial guess = minao
damping factor = 0
level_shift factor = 0
DIIS = <class 'pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 8
diis_damp = 0
SCF conv_tol = 1e-09
SCF conv_tol_grad = None
SCF max_cycles = 50
direct_scf = False
chkfile to save SCF result = /tmp/tmpoyx55b8n
max_memory 4000 MB (current use 390 MB)
Set gradient conv threshold to 3.16228e-05
init E= -152.062490646928
  HOMO = -0.462397775076163  LUMO = 0.16388011565275
cycle= 1 E= -152.06249064693  delta_E= -2.27e-12  |g|= 7.6e-07  |ddm|= 2.32e-06
  HOMO = -0.462397692137076  LUMO = 0.163880138617831
Extra cycle  E= -152.06249064693  delta_E= -1.14e-13  |g|= 3.67e-07  |ddm|= 1.11e-06
converged SCF energy = -152.06249064693


np.float64(-152.06249064693026)

In [69]:
def check_mo_span(mf, mo1, mo2):
    # _,actocc,actvir,_ = lnocc.split_mo()
    nocc = np.count_nonzero(mf.mo_occ)
    s1e = mf.get_ovlp()
    mo1occ = mo1[:,:nocc]
    mo1vir = mo1[:,nocc:]
    mo2occ = mo2[:,:nocc]
    mo2vir = mo2[:,nocc:]
    olp_occ = mo1occ.T @ s1e @ mo2occ
    olp_vir = mo1vir.T @ s1e @ mo2vir
    occspanerr = abs(olp_occ.T @ olp_occ - np.eye(olp_occ.shape[1])).max()
    virspanerr = abs(olp_vir.T @ olp_vir - np.eye(olp_vir.shape[1])).max()
    if occspanerr > 1e-10:
        raise ValueError(
            'LOs do not fully span the active occupied space! Max|<occ|LO><LO|occ>| = %e',
            occspanerr)
    if virspanerr > 1e-10:
        raise ValueError(
            'LOs do not fully span the active virtual space! Max|<vir|LO><LO|vir>| = %e',
            virspanerr)
    return occspanerr < 1e-10 , virspanerr < 1e-10

In [70]:
check_mo_span(mf, mf.mo_coeff, lno_coeff)

(np.True_, np.True_)

In [62]:
nocc = np.count_nonzero(mf.mo_occ)
actfrag = np.array([i for i in range(mol.nao) if i not in lno_frozen])
actocc = np.array([i for i in range(nocc) if i in actfrag])
actvir = np.array([i for i in range(nocc, mol.nao) if i in actfrag])

In [63]:
actocc

array([5, 6, 7, 8, 9])

In [64]:
lno_actocc = lno_coeff[:,actocc]
print(lno_actocc.shape)

(48, 5)


In [67]:
s1e = mlno.s1e
m = lno_actocc.T @ s1e @ orbloc
print(m)

[[ 1.58218691e-02 -8.51357575e-01 -6.56739645e-02 -1.26356067e-01
  -6.17615768e-16]
 [ 7.71660920e-04  1.83159359e-02 -6.92573858e-01  3.41823066e-01
   1.79814930e-16]
 [-7.20653120e-03 -1.67340792e-01 -9.32922029e-02  4.24796280e-01
   1.84108371e-14]
 [-9.19366496e-03 -2.60612335e-01  4.93956006e-01  7.05404835e-01
  -1.12659448e-15]
 [-1.51258062e-16 -2.06498501e-15 -2.37920035e-15  6.84996222e-15
  -9.99989698e-01]]


In [66]:
print(uocc_loc)

[[ 1.58218691e-02 -8.51357575e-01 -6.56739645e-02 -1.26356067e-01
  -6.17523335e-16]
 [ 7.71660920e-04  1.83159359e-02 -6.92573858e-01  3.41823066e-01
   1.79891587e-16]
 [-7.20653120e-03 -1.67340792e-01 -9.32922029e-02  4.24796280e-01
   1.84094225e-14]
 [-9.19366496e-03 -2.60612335e-01  4.93956006e-01  7.05404835e-01
  -1.12658431e-15]
 [-1.51231698e-16 -2.06489692e-15 -2.37903560e-15  6.84993318e-15
  -9.99989698e-01]]


In [42]:
mo_of, mo_oa, mo_va, mo_vf = mlno.split_mo_coeff()
print(mo_of.shape, mo_oa.shape, mo_va.shape, mo_vf.shape)

(48, 2) (48, 8) (48, 38) (48, 0)


In [47]:
s1e = mlno.s1e
uocc_loc = orbloc.T.conj() @ s1e @ orbocc # <loc|actocc>
print(uocc_loc.shape)
print(uocc_loc)

(5, 8)
[[-1.75263698e-03 -1.57286848e-02 -1.71303943e-17 -5.68532662e-04
  -4.36839283e-03 -9.97192324e-03 -4.25158449e-03  1.42379462e-16]
 [ 5.58944369e-02  8.49661212e-01 -3.90312782e-18 -1.32429591e-02
  -9.30155188e-02 -2.63482355e-01 -1.33297960e-01  2.42240581e-15]
 [ 1.26350836e-02  6.48051655e-02 -2.01618236e-15  6.99097458e-01
  -1.14598887e-01  2.87918237e-01  3.84110873e-01 -8.88584996e-16]
 [ 9.37823016e-03  1.26004800e-01  1.13320811e-15 -3.51157454e-01
   2.44408424e-01  6.76551969e-01  3.92583701e-01 -8.83905986e-15]
 [-2.07191035e-16  4.96401965e-16  7.09296934e-03 -3.16131669e-15
   2.95466776e-15  9.76950183e-15  1.64275603e-15  9.99964542e-01]]


In [46]:
from pyscf.lno.lno import projection_construction

In [ ]:
# (M^dagger M)u = eu 
# M = <loc|canactocc>
# u|canactocc> => orbtial in/out the space spanned by |loc>
uocc_loc, uocc_std, uocc_orth = \
            projection_construction(uocc_loc, mlno.lo_proj_thresh, mlno.lo_proj_thresh_active)

In [49]:
print(uocc_loc.shape)
print(uocc_loc)

(8, 4)
[[-4.12411866e-02  4.72035569e-02  2.60432001e-02  3.25898133e-13]
 [-6.64578616e-01  5.85679682e-01  4.54320607e-01  5.68111178e-12]
 [-1.85295794e-15 -2.03691498e-15  8.87680168e-14 -7.09304242e-03]
 [ 6.26187145e-01  7.05542708e-01 -4.14986609e-02 -5.15468337e-13]
 [-2.11218838e-01 -1.69209990e-01 -1.77770273e-01 -2.22609292e-12]
 [-3.27499844e-01  1.52344143e-01 -7.22456307e-01 -9.04492840e-12]
 [-1.12492406e-01  3.24221935e-01 -4.87490527e-01 -6.09802542e-12]
 [ 6.63550135e-15  6.45989898e-16  1.25136399e-11 -9.99974844e-01]]


In [50]:
print(uocc_loc.shape, uocc_std.shape, uocc_orth.shape)

(8, 4) (8, 1) (8, 3)


In [ ]:
from functools import reduce

def make_las(mlno, eris, orbloc, lno_type, lno_param):
    s1e = mlno.s1e

    orboccfrz_core, orbocc, orbvir, orbvirfrz_core = mlno.split_mo_coeff() # canonical
    moeocc, moevir = mlno.split_mo_energy()[1:3]

    ''' Projection of LO onto occ and vir
    '''
    uocc_loc = reduce(np.dot, (orbloc.T.conj(), s1e, orbocc)) # <loc|actocc>
    uocc_loc, uocc_std, uocc_orth = \
            projection_construction(uocc_loc, mlno.lo_proj_thresh, mlno.lo_proj_thresh_active)

    uvir_loc = reduce(np.dot, (orbloc.T.conj(), s1e, orbvir))
    uvir_loc, uvir_std, uvir_orth = \
            projection_construction(uvir_loc, mlno.lo_proj_thresh, mlno.lo_proj_thresh_active)
    # log.info('LO vir proj: %d active | %d standby | %d orthogonal',
    #          *[u.shape[1] for u in [uvir_loc,uvir_std,uvir_orth]])
    if uvir_loc.shape[1] == 0:
        uvir_loc = uvir_std = uvir_orth = None

    ''' LNO construction
    '''
    dmoo = mlno.make_lo_rdm1_occ(eris, moeocc, moevir, uocc_loc, uvir_loc, lno_type[0])
    if mlno._match_oldcode: dmoo *= 0.5 # TO MATCH OLD LNO CODE
    dmoo = reduce(np.dot, (uocc_orth.T.conj(), dmoo, uocc_orth))
    if lno_param[0]['norb'] is not None:
        lno_param[0]['norb'] -= uocc_loc.shape[1] + uocc_std.shape[1]
    uoccact_orth, uoccfrz_orth = natorb_select(dmoo, uocc_orth, **lno_param[0])
    orboccfrz = np.hstack((orboccfrz_core, np.dot(orbocc, uoccfrz_orth)))
    uoccact = subspace_eigh(np.diag(moeocc), np.hstack((uoccact_orth, uocc_std, uocc_loc)))[1]
    orboccact = np.dot(orbocc, uoccact)
    uoccact_loc = np.linalg.multi_dot((orboccact.T.conj(), s1e, orbloc))
    cput1 = log.timer_debug1('make_lo_rdm1_occ', *cput1)

    dmvv = mlno.make_lo_rdm1_vir(eris, moeocc, moevir, uocc_loc, uvir_loc, lno_type[1])
    if mlno._match_oldcode: dmvv *= 0.5 # TO MATCH OLD LNO CODE
    if uvir_orth is not None:
        dmvv = reduce(np.dot, (uvir_orth.T.conj(), dmvv, uvir_orth))
        if lno_param[1]['norb'] is not None:
            lno_param[1]['norb'] -= uvir_loc.shape[1] + uvir_std.shape[1]
        uviract_orth, uvirfrz_orth = natorb_select(dmvv, uvir_orth, **lno_param[1])
        orbvirfrz = np.hstack((np.dot(orbvir, uvirfrz_orth), orbvirfrz_core))
        uviract = subspace_eigh(np.diag(moevir), np.hstack((uviract_orth, uvir_std, uvir_loc)))[1]
        orbviract = np.dot(orbvir, uviract)
    else:
        orbviract, orbvirfrz = natorb_select(dmvv, orbvir, **lno_param[1])
        orbvirfrz = np.hstack((orbvirfrz, orbvirfrz_core))
        uviract = reduce(np.dot, (orbvir.T.conj(), s1e, orbviract))
        uviract = subspace_eigh(np.diag(moevir), uviract)[1]
        orbviract = np.dot(orbvir, uviract)
    cput1 = log.timer_debug1('make_lo_rdm1_vir', *cput1)

    ''' LAS construction
    '''
    orbfragall = [orboccfrz, orboccact, orbviract, orbvirfrz]
    orbfrag = np.hstack(orbfragall)
    norbfragall = np.asarray([x.shape[1] for x in orbfragall])
    locfragall = np.cumsum([0] + norbfragall.tolist()).astype(int)
    frzfrag = np.concatenate((
        np.arange(locfragall[0], locfragall[1]),
        np.arange(locfragall[3], locfragall[4]))).astype(int)
    frag_msg = '%d/%d Occ | %d/%d Vir | %d/%d MOs' % (
                    norbfragall[1], sum(norbfragall[:2]),
                    norbfragall[2], sum(norbfragall[2:4]),
                    sum(norbfragall[1:3]), sum(norbfragall)
                )
    if len(frzfrag) == 0:
        frzfrag = 0

    return orbfrag, frzfrag, uoccact_loc, frag_msg

In [ ]:
run_lnoccsd(
        mf = mycc._scf,
        lo_coeff = lo_iao, 
        frag_lolist = frag_iao_list,
        nfrozen = mycc.frozen, 
        thresh = 3e-5, 
        run_frg_list = None,
        atom_group = moliao.elements
        )

Number of LNO-FRAGMENT: 6


=========================== RUNNING LNO-FRAGMENT 1/6 ===========================
Center Atom O
number of active occupied orbitals: 5
number of active virtual orbitals: 24
Locating Fragment [0, 1, 2, 3, 4] by AOs
AOs with contribution > 0.01
        AO Label     Amp
      0 O 1s      0.9019
      0 O 2px     0.5103
      0 O 2py     0.4829
      0 O 2pz     0.4539
      0 O 2s      0.4206
      0 O 3pz     0.4033
      0 O 3py     0.3156
      0 O 3s      0.3042
      0 O 3px     0.2529
      2 H 2px     0.2257
      1 H 2py     0.2102
      1 H 2px     0.0825
      1 H 1s      0.0811
      2 H 1s      0.0769
      1 H 2pz     0.0722
      2 H 2pz     0.0700
      2 H 2py     0.0626
      2 H 2s      0.0360
      1 H 2s      0.0332
E(MODIFIED_CCSD) = -152.2904073592697  E_corr = -0.2279167123418017
LNO-MP2 Orbital Energy: -0.16955726
LNO-CCSD Orbital Energy: -0.17710627


=========================== RUNNING LNO-FRAGMENT 2/6 ===========================
Center 

(np.float64(-0.403856347365407), np.float64(-0.422581591670929), np.int32(30))